# 1장 AI 에이전트 소개

TinyAgent의 기본 구조를 정의합니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 1장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다.

Colab에서 GPU를 사용하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택합니다.

In [ ]:
# %%capture
# !pip install illustrated-agents

---

## 1. 에이전트의 구성 요소

이 책에서는 언어 모델(LLM)의 API를 호출하는 코드부터 작성해 `TinyAgent`를 구현합니다. 1장에서 `agent.py`에 기본 클래스를 정의하고, 이후 각 장에서 모듈을 하나씩 연결해 에이전트가 수행할 수 있는 작업을 늘립니다. 전체 구현은 [소스 폴더](../src/illustrated_agents/)에서 확인할 수 있습니다.

![TinyAgent의 장별 구성 요소](../images/overview.png)

| 장 | 추가하는 모듈 | 실습 내용 |
|---|---|---|
| 1 | 기본 클래스 | 나중에 구현할 메서드와 속성 정의 |
| 2 | `llm.py` | 대화 상태를 저장하지 않고 언어 모델 호출 |
| 3 | 추가 모듈 없음 | 프롬프트를 이용한 추론과 모델 자체의 추론 |
| 4 | `memory.py` | 대화 기록 저장 |
| 5 | `tools.py`, `toolbox.py` | 프롬프트 또는 모델의 내장 기능으로 도구 호출 |
| 6 | `planning.py` | ReAct 반복문으로 여러 단계에 걸쳐 추론 및 행동 |
| 8 | 조정자(coordinator) | 여러 에이전트의 협업 |

최근 모델 중에는 추론과 도구 호출을 하도록 학습된 모델이 있습니다. 이 기능을 직접 구현해 보면 모델이 도구 이름과 인자를 어떤 형식으로 출력해야 하는지, 프로그램은 그 출력을 어떻게 읽고 실행하는지 살펴볼 수 있습니다. 예를 들어 여러 줄의 텍스트를 도구에 전달할 때 JSON과 XML의 차이가 드러납니다.

![Gemma 3의 프롬프트 방식과 Gemma 4의 내장 기능 비교](../images/evolution.png)

먼저 Gemma 3에 프롬프트로 추론 과정과 도구 호출 형식을 지정합니다. 이후 같은 작업을 Gemma 4의 내장 추론 및 도구 호출 기능으로 수행해 두 방식을 비교합니다. 다른 공개 가중치 모델이나 상용 모델도 예제의 연결 설정을 바꿔 사용할 수 있습니다.

## 2. `TinyAgent` 클래스

먼저 `TinyAgent`의 기본 구조를 정의합니다. 이후 언어 모델, 메모리, 도구와 계획 기능을 이 클래스에 연결합니다.

![1장에서 정의하는 TinyAgent의 기본 구조](../images/ch1.png)

언어 모델을 연결하기 전에도 다음 항목을 정의할 수 있습니다.

- 각 장에서 구현할 구성 요소를 저장하는 속성
- 사용자 작업을 받아 에이전트를 실행하는 `run` 메서드
- 한 단계의 처리를 담당하는 `_step` 메서드. 한 작업에서 이 메서드를 여러 번 호출할 수 있습니다.

In [ ]:
class TinyAgent:
    """교육용 에이전트의 기본 클래스를 정의한다."""

    def __init__(self):
        self.llm = None  # 2~3장에서 언어 모델 연결
        self.memory = None  # 4장에서 메모리 추가
        self.tools = None  # 5장에서 도구 추가
        self.planner = None  # 6장에서 계획 기능 추가

    def run(self, task: str) -> str:
        """주어진 작업으로 에이전트를 실행한다."""
        return self._step(task)

    def _step(self, task: str) -> str:
        """에이전트의 한 단계를 실행한다."""
        # 이후 장에서 구현할 부분
        return f"Received: {task}"

    def _execute_action(self, action: str) -> str:
        """도구가 수행할 동작을 실행한다."""
        # 이후 장에서 구현할 부분
        return f"Executed action: {action}"

아래 코드 설명에서 작업 전체를 실행하는 `run`과 한 단계를 처리하는 `_step`의 호출 관계를 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch1 import agent_annotated; agent_annotated

에이전트를 실행해 봅니다. 현재 `_step`은 받은 질문을 그대로 응답에 포함하므로 덧셈을 계산하지 않습니다.

In [ ]:
agent = TinyAgent()
agent.run("What is 2 + 2?")

다음 장에서 언어 모델을 연결하면 에이전트가 질문에 답할 수 있습니다.

---

# 구현한 내용

각 장의 마지막에서는 새로 만들거나 수정한 코드를 확인합니다. 전체 예제는 [소스 폴더](../src/illustrated_agents/)에 있습니다. 아래는 수업에서 다루는 모듈 구성입니다.

  ```
  TinyAgent
  ├── __init__.py
  ├── agent.py
  ├── cli.py
  ├── display.py
  ├── llm.py
  ├── memory.py
  ├── planning.py
  ├── skills.py
  ├── toolbox.py   
  ├── tools.py
  ├── trajectory.py
  └── utils.py
  ```

이후 각 모듈을 구현하면서 에이전트에서 쓰이는 방식을 살펴봅니다. 이번 장에서는 `TinyAgent`의 기본 구조를 정의했습니다. 다음 셀을 실행해 구현 내용을 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch1 import what_we_built; what_we_built

# 다음 장

[2장](../chapter02/chapter02.ipynb)에서는 언어 모델의 동작과 추론 서비스 선택 방법을 살펴보고, `TinyAgent`가 모델을 호출하도록 구현합니다.